In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import os
import pickle

In [ ]:
def product_vectors_comp():
    csv_path = os.path.join("..","..","Task 1", "mongodb", "product_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    tfidf = TfidfVectorizer(stop_words='english')
    
    # Keep it as a SPARSE matrix (Do not use .toarray())
    vectors = tfidf.fit_transform(df['combined'].fillna(''))

    #use a pkl
    with open("tfidf_product.pkl", "wb") as f:
        pickle.dump(tfidf, f)

    scipy.sparse.save_npz("master_product_vectors_TFIDF.npz", vectors)
    
    df_ids = pd.DataFrame({'id': df['parent_asin'].astype(str)})
    df_ids.to_parquet("master_product_ids_TFIDF.parquet", engine="pyarrow")
    
    print("TFIDF sparse matrix, IDs, and model saved")
product_vectors_comp()



In [ ]:
def product_vectors():
    csv_path = os.path.join("..","..","Task 1", "mongodb", "product_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    #if you dont set a max_features, the file gets too big at around 68.8Gb for just this small dataset
    tfidf = TfidfVectorizer(max_features=256, stop_words=None)
    
    #convert matrix into list of lists
    vectors = tfidf.fit_transform(df['combined'].fillna('')).toarray()
    vectors2 = tfidf.fit_transform(df['Small'].fillna('')).toarray()

    df_out = pd.DataFrame({
        'id': df['parent_asin'].astype(str),
        'vector1': list(vectors),
        'vector2': list(vectors2),
        'combined': df['combined'],
        'Small': df["Small"],
        "title": df["title"].astype(str)
    })

    df_out.to_parquet("master_product_vectors_TFIDF.parquet", engine="pyarrow")

product_vectors()

In [ ]:
def user_vectors_comp():
    csv_path = os.path.join("Task 1", "mongodb", "user_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    tfidf = TfidfVectorizer(stop_words="english")
    
    vectors = tfidf.fit_transform(df['item_string'].fillna('')).toarray()
    
    # SAVE THE MODEL
    with open("tfidf_user.pkl", "wb") as f:
        pickle.dump(tfidf, f)
    
    scipy.sparse.save_npz("master_user_vectors_TFIDF.npz", vectors)
    
    df_out = pd.DataFrame({
        'id': df['_id'].astype(str),
        'metadata': [{'user_id': str(uid)} for uid in df['_id']]
    })
    
    df_out.to_parquet("master_user_vectors_TFIDF.parquet", engine="pyarrow")
    print("User vectors and model saved.")

user_vectors_comp()

In [ ]:
def user_vectors():
    csv_path = os.path.join("Task 1", "mongodb", "user_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    tfidf = TfidfVectorizer(max_features=256, stop_words=None)
    
    #convert matrix into list of lists
    vectors = tfidf.fit_transform(df['item_string'].fillna('')).toarray()
    
    df_out = pd.DataFrame({
        'id': df['_id'].astype(str),
        'vector': list(vectors),
        'metadata': [{'user_id': str(uid)} for uid in df['_id']]
    })
    
    df_out.to_parquet("master_user_vectors_TFIDF.parquet", engine="pyarrow")

user_vectors()